In [ ]:
import pandas as pd
from pathlib import Path

# --- 1) Caminhos
DATA_INTERIM = Path("../data/interim")
DATA_ANALYSIS = Path("../data/analysis")
DATA_ANALYSIS_IPV4 = Path("../data/analysis/ipv4")
DATA_ANALYSIS_IPV6 = Path("../data/analysis/ipv6")


In [ ]:
full = pd.read_parquet(DATA_INTERIM / "full_resolved_clean.parquet")

# count diferent unique as_org

print("unique as_org:", full["as_org"].nunique())

# count concentration of each as_org in the dataset
print("as_org concentration:")
print(full["as_org"].value_counts().head(10))  # top 10 as_org by number of companies
print("-------------")

# print percentage of total companies for each as_org
print("as_org concentration percentage:")
print(full["as_org"].value_counts(normalize=True).head(10) * 100)  # top 10 as_org by percentage of total companies

unique as_org: 1287
as_org concentration:
as_org
Cloudflare, Inc.                         22549
Amazon.com, Inc.                         10509
Akamai International B.V.                 5900
Microsoft Corporation                     4163
Host Europe GmbH                          3626
Vercara, LLC                              3405
Tiggee LLC                                2208
Hangzhou Alibaba Advertising Co.,Ltd.     1921
Alibaba (US) Technology Co., Ltd.         1561
NSONE Inc                                 1323
Name: count, dtype: int64

as_org concentration percentage:
as_org
Cloudflare, Inc.                         29.513239
Amazon.com, Inc.                         13.754695
Akamai International B.V.                 7.722210
Microsoft Corporation                     5.448739
Host Europe GmbH                          4.745887
Vercara, LLC                              4.456631
Tiggee LLC                                2.889939
Hangzhou Alibaba Advertising Co.,Ltd.     2.514299
Alibab

In [5]:
import pandas as pd

def hhi(df, grupo, peso=None, escala=10_000):
    """HHI de df agrupado por `grupo` (ex.: 'dns_provider_group' ou 'org_name').
    peso=None  -> participação por contagem de linhas (empresas)
    peso='col' -> participação pela soma de uma coluna (ex.: receita, market cap)."""
    if peso is None:
        valores = df.groupby(grupo).size()            # nº de linhas por grupo
    else:
        valores = df.groupby(grupo)[peso].sum()       # soma do peso por grupo
    valores = valores[valores > 0]                    # grupos sem peso não entram

    n = len(valores)
    if n == 0:                                        # tabela vazia: evita divisão por zero
        return {"hhi": None, "hhi_norm": None, "n_grupos": 0, "top1_share": None, "top3_share": None}

    shares = valores / valores.sum()                  # participações somam 1
    h = float((shares ** 2).sum())                    # HHI em fração (0 a 1)

    # normalizado: (H - 1/N) / (1 - 1/N); com 1 grupo só, definimos como 1 (concentração total)
    h_norm = (h - 1 / n) / (1 - 1 / n) if n > 1 else 1.0

    ordenado = shares.sort_values(ascending=False)
    return {"hhi": h * escala, 
            "hhi_norm": h_norm, 
            "n_grupos": n,
            "top1_share": float(ordenado.iloc[0]),    # fatia do maior provedor
            "top3_share": float(ordenado.iloc[:3].sum())}  # CR3, métrica clássica ao lado do HHI

In [6]:
full_org_deduplicated = full.drop_duplicates(subset=["ticker_symbol", "ip_type", "as_org"], keep="first")
full_org_deduplicated.to_parquet(DATA_ANALYSIS / "full_org_deduplicated.parquet", index=False)
full_org_deduplicated.to_csv(DATA_ANALYSIS / "full_org_deduplicated.csv", index=False)

full_ipv4_org_deduplicated = full_org_deduplicated[full_org_deduplicated["ip_type"] == "A"]
full_ipv4_org_deduplicated.to_parquet(DATA_ANALYSIS / "full_ipv4_org_deduplicated.parquet", index=False)
full_ipv4_org_deduplicated.to_csv(DATA_ANALYSIS / "full_ipv4_org_deduplicated.csv", index=False)

full_ipv6_org_deduplicated = full_org_deduplicated[full_org_deduplicated["ip_type"] == "AAAA"]
full_ipv6_org_deduplicated.to_parquet(DATA_ANALYSIS / "full_ipv6_org_deduplicated.parquet", index=False)
full_ipv6_org_deduplicated.to_csv(DATA_ANALYSIS / "full_ipv6_org_deduplicated.csv", index=False)

#unique as_org, count of companies per as_org, percentage of total companies per as_org, total revenue per as_org, percentage of total revenue per as_org, hhi
as_org_counts = full_org_deduplicated.groupby("as_org").agg(
    company_count=("ticker_symbol", "nunique"),
    total_revenue=("revenue_ttm", "sum")
).reset_index()

# Calculate percentages
as_org_counts["company_percentage"] = (as_org_counts["company_count"] / as_org_counts["company_count"].sum()) * 100
as_org_counts["revenue_percentage"] = (as_org_counts["total_revenue"] / as_org_counts["total_revenue"].sum()) * 100
as_org_counts.to_parquet(DATA_ANALYSIS / "as_org_counts.parquet", index=False)

print("Top ASN Organizations by company count")
display(as_org_counts.sort_values(by="company_count", ascending=False).head(5))

print("Top ASN Organizations by company revenue")
display(as_org_counts.sort_values(by="total_revenue", ascending=False).head(5))

Top ASN Organizations by company count


,as_org,company_count,total_revenue,company_percentage,revenue_percentage
250,"Cloudflare, Inc.",2302,15136332915574,17.376208,9.294658
68,"Amazon.com, Inc.",1462,13323154253110,11.035628,8.181253
513,Host Europe GmbH,920,2758400728408,6.944444,1.693831
715,Microsoft Corporation,595,7252810149323,4.491244,4.453681
1174,"Vercara, LLC",588,13335123511912,4.438406,8.188603


Top ASN Organizations by company revenue


,as_org,company_count,total_revenue,company_percentage,revenue_percentage
50,Akamai International B.V.,519,22143777292133,3.917572,13.597669
250,"Cloudflare, Inc.",2302,15136332915574,17.376208,9.294658
1174,"Vercara, LLC",588,13335123511912,4.438406,8.188603
68,"Amazon.com, Inc.",1462,13323154253110,11.035628,8.181253
715,Microsoft Corporation,595,7252810149323,4.491244,4.453681


In [ ]:
# todas as as_org únicas (com nº de empresas e receita), ordenadas por nº de empresas
as_orgs = (
    full.dropna(subset=["as_org"])
        .drop_duplicates(["ticker_symbol", "as_org"])      # 1 empresa conta 1 vez por org
        .groupby("as_org")
        .agg(empresas=("ticker_symbol", "nunique"),
             receita=("revenue_ttm", "sum"),
             pais=("as_country_code", lambda s: s.mode().iat[0] if s.notna().any() else None))
        .sort_values("empresas", ascending=False)
        .reset_index()
)

print("as_org únicas:", len(as_orgs))
as_orgs.to_csv("as_orgs_unicas.csv")



as_org únicas: 1287
